# HybridLLMRouter - Training

This notebook demonstrates how to train the **HybridLLMRouter**.

## Overview

HybridLLMRouter routes between small and large models using an MLP regressor.
It supports different routing modes: deterministic, probabilistic, and transformed.

**Key Features**:
- Binary routing (small vs large model)
- MLP-based decision making
- Configurable routing threshold

## 1. Environment Setup

In [ ]:
# Install required packages (for Colab)
!pip install llmrouter-lib transformers torch
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
from llmrouter.models.hybrid_llm import HybridLLMRouter, HybridLLMTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

## 2. Configuration

HybridLLMRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `router_mode` | Routing mode | "probabilistic" |
| `router_tau` | Temperature for probabilistic routing | 0.1 |
| `router_threshold` | Decision threshold | 0.5 |
| `hidden_layer_sizes` | MLP architecture | [128, 64] |

In [ ]:
import yaml

CONFIG_PATH = "configs/model_config_train/hybrid_llm.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

## 3. Initialize Router

In [ ]:
router = HybridLLMRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Router mode: {config.get('router_mode', 'probabilistic')}")
print(f"Threshold: {config.get('router_threshold', 0.5)}")

## 4. Training

In [ ]:
trainer = HybridLLMTrainer(router=router, device='cpu')

print("Trainer initialized!")

In [ ]:
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

## 5. Model Verification

In [ ]:
# Test prediction
test_query = {"query": "What is the capital of France?"}
result = router.route_single(test_query)

print(f"Test query: {test_query['query']}")
print(f"Routed to: {result['model_name']}")

## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up HybridLLMRouter with YAML configuration
2. **Trained Model**: Learned routing decision boundary
3. **Verified Model**: Tested routing with sample queries

**Routing Modes**:
- **deterministic**: Hard threshold decision
- **probabilistic**: Soft probability-based routing
- **transformed**: Temperature-scaled probabilities

**Next Steps**:
- Use the next part of notebook for inference

# HybridLLMRouter - Inference

This notebook demonstrates how to use a trained **HybridLLMRouter** for inference.

## 1. Environment Setup (optional)

In [ ]:
# Install required packages (for Colab)
!pip install llmrouter-lib transformers torch
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
from llmrouter.models.hybrid_llm import HybridLLMRouter
from llmrouter.utils import setup_environment
import yaml

setup_environment()

## 2. Load Trained Router

In [ ]:
CONFIG_PATH = "configs/model_config_train/hybrid_llm.yaml"

router = HybridLLMRouter(yaml_path=CONFIG_PATH)
print("Router loaded!")

## 3. Query Routing

In [ ]:
EXAMPLE_QUERIES = [
    {"query": "What is 2 + 2?"},
    {"query": "Explain quantum entanglement."},
    {"query": "Write a sorting algorithm."},
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

## 4. File-Based Inference

Load queries from a file and save results.

In [ ]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/hybrid_llm_router_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

## Summary

HybridLLMRouter provides:
- Efficient binary routing between small and large models
- Configurable routing modes for different use cases